# Reasoning-trace generation, independent-solve variant (teacher: gemma-4-31B-it, via API)

Called through the OpenAI-compatible FPT Cloud API (same as the baseline notebooks), not run
locally via vLLM -- much cheaper/faster than the Qwen3-Next-80B-A3B-Thinking notebook this was
copied from, and now the better teacher on this task besides: a 100-sample trial with the
convention-hardened prompt below (v3: R2/R7 fixed, R8/R9 added for gemma's specific failure
modes -- quoted row labels, bare-number `<program>` blocks) reached **80.0% exact match**,
beating the 80B teacher's 65.0% on the same trial.

**Now a full production run** (`SAMPLE_LIMIT = None`), not a comparison trial -- this generates
the actual `train_with_reasoning_trace.json` to feed into the SFT notebooks
(`sft-w-reasoning-trace-distill/`), replacing whichever reasoning-trace source they currently use.

Run strategy: `N_SAMPLES = 1` for the full pass (79/80 matches on the trial landed on the first
attempt, so a single try is already close to the ceiling and ~5x cheaper than sampling multiple
attempts upfront). If the yield on the full set is meaningfully lower than 80%, bump `N_SAMPLES`
to 3 and re-run the same generation cell -- it only spends new calls on samples still short of
that count, so failures from pass 1 are the only thing that costs anything in pass 2.

In [15]:
import os
from pathlib import Path
from dotenv import load_dotenv

# API-based teacher, not a local vLLM load -- same .env used by every other
# API notebook in this repo (API_KEY / BASE_URL, FPT Cloud endpoint).
load_dotenv()
API_KEY = os.environ["API_KEY"]
BASE_URL = os.environ["BASE_URL"]
MODEL = "gemma-4-31B-it"

print(f"Teacher model: {MODEL}")
print(f"Base URL: {BASE_URL}")


Teacher model: gemma-4-31B-it
Base URL: https://mkp-api.fptcloud.com/v1


In [16]:
%%capture
!pip install -q openai python-dotenv tabulate


In [ ]:
import json
from pathlib import Path

# Robust to whichever working directory Jupyter starts in (repo root vs.
# the notebook's own folder), same pattern as the other notebooks in this repo.
_CANDIDATES = [
    Path("datasets/ViNumQA/train.json"),
    Path("../../../datasets/ViNumQA/train.json"),
]
TRAIN_JSON_PATH = next((p for p in _CANDIDATES if p.exists()), None)
if TRAIN_JSON_PATH is None:
    raise FileNotFoundError(
        f"Could not find train.json. cwd={Path.cwd()}, tried: {[str(p) for p in _CANDIDATES]}"
    )

OUTPUT_DIR = Path("outputs/conr_trace_gemma_independent_solve")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_data = json.load(open(TRAIN_JSON_PATH, encoding="utf-8"))
print(f"Loaded {len(train_data)} samples from {TRAIN_JSON_PATH}")


## Evidence-type classification (for reporting, not sampling)

Classifies each sample by which evidence type its gold program's numeric
arguments come from (table values vs. text values vs. both). Unlike the
150-sample trial, this run uses every sample in `train.json` -- the
classification is kept only so the per-category breakdown can be reported
alongside the overall exact-match rate.

In [18]:
import re

def numbers_in(text):
    return set(re.findall(r"-?\d+\.?\d*", text))

def classify_evidence_type(sample):
    prog = sample["qa"]["program"]
    pre_text = " ".join(sample["pre_text"])
    post_text = " ".join(sample["post_text"])
    text_nums = numbers_in(pre_text) | numbers_in(post_text)
    table_flat = " ".join(" ".join(row) for row in sample["table"])
    table_nums = numbers_in(table_flat)
    prog_nums = set(re.findall(r"-?\d+\.?\d*", prog))

    uses_text = bool(prog_nums & text_nums)
    uses_table = bool(prog_nums & table_nums)

    if uses_table and not uses_text:
        return "table_only"
    elif uses_text and not uses_table:
        return "text_only"
    elif uses_text and uses_table:
        return "table_text"
    return "unclassified"

buckets = {"table_only": [], "table_text": [], "text_only": [], "unclassified": []}
for i, s in enumerate(train_data):
    buckets[classify_evidence_type(s)].append(i)

for k, v in buckets.items():
    print(f"{k}: {len(v)}")

# Full production run. The prompt was validated in two stages on a 100-sample
# trial: v2 reached 56-61% exact match, then v3 (R2/R7 fixed, R8/R9 added to
# target gemma's specific failure modes -- quoted row labels, bare-number
# <program> blocks) reached 80.0%, beating the 80B teacher's 65.0% on the
# same trial. Cleared to None to run every sample in train.json.
SAMPLE_LIMIT = None
sample_idx = list(range(len(train_data)))
if SAMPLE_LIMIT is not None:
    sample_idx = sample_idx[:SAMPLE_LIMIT]
print(f"\nTotal samples to process: {len(sample_idx)} (SAMPLE_LIMIT={SAMPLE_LIMIT})")

table_only: 286
table_text: 141
text_only: 54
unclassified: 103

Total samples to process: 584 (SAMPLE_LIMIT=None)


## Context formatting (same as the 0-shot/1-shot/few-shot/SFT notebooks)

In [19]:
from tabulate import tabulate

def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

samples = []
for i in sample_idx:
    s = train_data[i]
    samples.append({
        "train_index": i,
        "evidence_type": classify_evidence_type(s),
        "pre_text": formatting_pre_text(s),
        "table": formatting_table(s),
        "post_text": formatting_post_text(s),
        "question": s["qa"]["question"],
        "program": s["qa"]["program"],
        "answer": s["qa"]["exe_ans"],
    })

print(f"Prepared {len(samples)} samples for trace generation.")
samples[0]

Prepared 584 samples for trace generation.


{'train_index': 0,
 'evidence_type': 'text_only',
 'pre_text': 'vay nợ, phải thu và hàng tồn kho tiếp tục tăng cao: phải thu tăng mạnh gần 50%  lên mức 8.344 tỷ đồng, trong đó chủ yếu là phải thu ngắn hạn khác (5.925 tỷ đồng).\nhàng tồn kho cũng tăng mạnh 15% so với đầu năm lên 5.310 tỷ đồng với 18 dự án bất  động sản dở dang.\ntổng nợ ngắn hạn và dài hạn của dxg cũng tăng lên 3.401 tỷ đồng  (tăng 16% so với năm trước).\ndòng tiền kinh doanh 3 quý vừa qua âm 100 tỷ: hai năm trước dxg cũng liên tục bị  âm dòng tiền khoảng 1.000 tỷ mỗi năm.\ndo đó để bổ sung vốn lưu động và đầu tư dự  án, dxg phải tiếp tục huy động vốn thông qua phát hành trái phiếu.\ntính đến hết quý  3, tổng giá trị trái phiếu của dxg đạt 2.102 tỷ đồng (tăng gần 4 lần so với đầu năm),  có kỳ hạn từ 2 – 5 năm.\nđiều này dẫn đến áp lực về chi phí lãi vay lớn.\nvẫn tồn tại vướng mắc đối với các dự án liên quan đến petroland: hai dự án được  chuyển nhượng từ petroland là dự án chung cư thăng long và chuyển nhượng cổ  phần 

## Independent-solve prompt (v2 -- convention-hardened)

The teacher gets only context + question, exactly what the student sees at
inference time. It must derive the program itself.

**v2 changes.** The v1 run reached only 14.4% exact match, but an error analysis
of all 2,993 attempts showed the failures were overwhelmingly *formatting*, not
reasoning: 37.6% nested calls inside arguments, 27.3% appended a
`multiply(#N, 100)` to turn a ratio into a percent, 9.3% wrapped a single value
in `table_sum(...)`, 9.0% invented operators such as `table_value(...)`, and 9.2%
never emitted a `<program>` at all (the model drifted into English self-talk and
ran out of tokens). Each rule below targets one measured failure mode, and the
conventions are stated as counts over the 2,993 gold programs so they are not
guesses:

- nested calls appear in **6 / 2993** gold programs -> effectively never
- `table_*` with a row label + `none` appears **454** times, with explicit
  numbers **2** times -> the row-label form is the convention
- gold keeps ratios as decimal fractions (`divide(180, 8012)`), never `* 100`

The prompt also now forbids writing the program inside the `<think>` prose,
because v1 traces that did so spelled out a *different* program than the one
being trained on.

In [20]:
CONR_SYSTEM_PROMPT = """You are a senior financial analyst solving a numerical question about a
Vietnamese financial document. You are given only the context (text before a
table, the table itself, text after the table) and the question -- nothing
else. You must derive the answer yourself.

Generate a sequential computation program to answer the question, using ONLY
the following 10 operators:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_label, none) -> sum of the numeric values in the table row named row_label
8. table_average(row_label, none) -> arithmetic mean of the numeric values in the table row named row_label
9. table_max(row_label, none) -> maximum of the numeric values in the table row named row_label
10. table_min(row_label, none) -> minimum of the numeric values in the table row named row_label

### PROGRAM FORMAT RULES (follow exactly -- these are the dataset's conventions):

R1. NEVER nest one operator inside another's arguments. Write each operation as
    its own step, separated by ", ", and refer to an earlier step's result with
    #0 (step 1), #1 (step 2), and so on.
      WRONG: divide(subtract(2438.4, 2408.8), 2408.8)
      RIGHT: subtract(2438.4, 2408.8), divide(#0, 2408.8)

R2. Almost always leave the answer as a decimal fraction -- do NOT append
    multiply(#N, 100). This applies even when the question uses words like
    "phan tram" / "ty le" / "chiem bao nhieu phan tram" / "hoan thanh bao
    nhieu phan tram" -- that wording does NOT reliably signal a *100 step in
    this dataset; the exact same phrasing occurs in gold programs that do
    and do not multiply by 100, so it cannot be used as a cue. Only append
    multiply(#N, 100) in the rare case where the question explicitly asks
    you to report a percentage-POINT change or completion rate AND you have
    strong independent evidence (not just the word "phan tram") that the
    dataset's stated answer is scaled to a 0-100 range rather than a 0-1
    fraction. When in doubt -- which is most of the time -- do NOT multiply
    by 100.
      RIGHT (default): divide(180, 8012)
      RIGHT (default): divide(700, 4477)
      RARE exception only, not the default: divide(700, 4477), multiply(#0, 100)

R3. Use ONLY the 10 operators above. Do not invent operators such as
    table_value(...), percent(...), sum(...) or lookup(...).

R4. When the question aggregates an ENTIRE row of the table (its total, average,
    max or min across all periods), reference the row by its label exactly as it
    appears in the table's first column, with `none` as the second argument --
    do not enumerate the row's values.
      WRONG: table_max(1584, 1261, 5786, 3428, 1479, 2290)
      RIGHT: table_max(EPS (VND), none)

R5. Never wrap a single value in a table operator; write the number itself.
      WRONG: subtract(table_sum(17005), table_sum(12207))
      RIGHT: subtract(17005, 12207)

R6. When you combine a few specific values that you read off individually
    (rather than a whole row), chain binary operators instead of using a table
    operator.
      WRONG: table_sum(6851, 9091, 14606)
      RIGHT: add(6851, 9091), add(#0, 14606)

R7. Write numbers as plain digits: drop currency symbols and thousand
    separators, keep the decimal point, and ALWAYS drop a trailing '%' sign
    too -- write the bare numeric value even when the source states it as a
    percentage. "3.564 ty" -> 3564, "$ 620,125" -> 620125, "15.1%" -> 15.1.
    If a needed value is missing, use 'none'.

R8. Never wrap a table row label in quotation marks -- write it exactly as it
    appears in the table, with no surrounding " " or ' '.
      WRONG: table_max("Lai rong", none)
      RIGHT: table_max(Lai rong, none)

R9. The <program> block must always contain one or more operator calls in the
    exact format shown above -- never a bare number, a bare word, or any text
    that isn't an operator(...) call.
      WRONG: <program>11565</program>
      RIGHT: <program>multiply(11228, 1.03)</program>

### OUTPUT FORMAT (strict):
<think>
[Your reasoning, IN VIETNAMESE, 3-6 sentences. Must:
 1. Identify which specific values are needed and WHERE they come from
    (quote the exact row/column label from the table, or the exact sentence
    from pre_text/post_text) -- never state a number without saying where it
    was found.
 2. Explain WHY each operator was chosen (e.g. "vi cau hoi yeu cau ty le giua
    hai ky nen ta chia gia tri nam sau cho nam truoc" rather than just naming
    the operator).
 3. If the calculation has multiple steps, walk through them in order,
    referring to intermediate results the way the program does (ket qua buoc 1,
    ket qua buoc 2, ...).]
</think>
<program>YOUR_DERIVED_PROGRAM</program>

### REASONING STYLE RULES:
S1. Write the reasoning in Vietnamese. Do not think out loud in English.
S2. Do NOT write the final program string (or any concrete operator call with
    real numbers, such as "divide(180, 8012)") inside the <think> block. The
    program belongs only in the <program> block. Naming an operator in prose
    (e.g. "ta dung phep divide") is fine.
S3. Commit to one derivation. Do not second-guess yourself, re-derive the
    calculation, or narrate doubts ("Wait", "Hmm", "Nhung ma", "Vay dung").
S4. Keep the <think> block to 3-6 sentences and stop. Do not pad with generic
    financial commentary unrelated to the calculation."""

CONR_USER_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### YOUR TASK:
Solve this independently -- you have not been given the program or the answer.
Write the <think>...</think> reasoning trace and <program> block as instructed."""

## Load the teacher (gemma-4-31B-it, via API)

In [21]:
from openai import OpenAI

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

# Empirical probe, same reasoning as the baseline notebooks: find out whether
# gemma-4-31B-it needs a large token budget for this task BEFORE committing
# to a budget, since writing a full <think> reasoning trace is a much longer
# ask than the terse baseline. Probes at a generous fixed budget, independent
# of whatever MAX_TOKENS ends up being, so a truncated probe still tells you
# something (rather than being circular).
PROBE_TOKENS = 4096
_probe_sample = train_data[0]
_probe_user = CONR_USER_FRAME.format(
    pre_text="\n".join(_probe_sample["pre_text"]),
    table=str(_probe_sample["table"]),
    post_text="\n".join(_probe_sample["post_text"]),
    question=_probe_sample["qa"]["question"],
)
_probe = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": CONR_SYSTEM_PROMPT},
        {"role": "user", "content": _probe_user},
    ],
    temperature=0.6,
    top_p=0.9,
    max_tokens=PROBE_TOKENS,
)
_choice = _probe.choices[0]
_content = _choice.message.content
print("finish_reason:", _choice.finish_reason)
print("content (first 500 chars):", repr((_content or "")[:500]))
_truncated = _choice.finish_reason == "length" and not (_content or "").strip()
if _truncated:
    print(f"\nWARNING: truncated with no content even at PROBE_TOKENS={PROBE_TOKENS} -- "
          f"this model needs a bigger budget (or is hitting a proxy timeout on long "
          f"generations, as happened with GLM-5.1/5.2). Check before running the full batch.")

MAX_TOKENS = 8192  # matches the 80B teacher's max_tokens, for a fair comparison


finish_reason: stop
content (first 500 chars): '<think>\nĐể tìm giá trị hàng tồn kho đầu năm, ta dựa vào thông tin trong phần văn bản trước bảng. Văn bản cho biết "hàng tồn kho cũng tăng mạnh 15% so với đầu năm lên 5.310 tỷ đồng". Điều này có nghĩa là giá trị hiện tại (5.310 tỷ đồng) bằng giá trị đầu năm cộng thêm 15% của giá trị đầu năm, hay giá trị hiện tại bằng 115% giá trị đầu năm. Vì vậy, ta dùng phép chia giá trị hiện tại cho 1.15 để tìm giá trị hàng tồn kho đầu năm.\n</think>\n<program>divide(5310, 1.15)</program>'


## Generate traces (parallel API calls, N_SAMPLES independent attempts each)

In [ ]:
import time
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# N_SAMPLES independent attempts per prompt. Pass 1 (N_SAMPLES=1, both splits)
# already ran and is seeded for free below via PREV_RESULTS_PATH -- this is
# pass 2: N_SAMPLES=3, which only spends new calls on samples still short of
# 3 attempts (i.e. every sample that pass 1 didn't already match), since
# matched samples are excluded from `work` entirely.
N_SAMPLES = 3
N_WORKERS = 8

GEN_CKPT_PATH = OUTPUT_DIR / f"gen_checkpoint_{TRAIN_JSON_PATH.stem}_n{N_SAMPLES}.json"
PREV_RESULTS_PATH = OUTPUT_DIR / f"independent_solve_trace_{TRAIN_JSON_PATH.stem}.json"


def build_prompt_messages(sample_dict):
    user_msg = CONR_USER_FRAME.format(
        pre_text=sample_dict["pre_text"], table=sample_dict["table"],
        post_text=sample_dict["post_text"], question=sample_dict["question"],
    )
    return [
        {"role": "system", "content": CONR_SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]


def call_teacher_once(sample_dict, max_retries=3):
    messages = build_prompt_messages(sample_dict)
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                temperature=0.6,
                top_p=0.9,
                max_tokens=MAX_TOKENS,
            )
            text = resp.choices[0].message.content
            if text and text.strip():
                return text.strip()
        except Exception as e:
            if attempt == max_retries:
                print(f"[attempt failed after {max_retries} tries] {e}")
                return ""
        if attempt < max_retries:
            time.sleep(2 * attempt)
    return ""


class _FakeCompletion:
    """Mimics vllm.CompletionOutput just enough (.text) for the parsing cell
    below, which was written against vLLM's RequestOutput.outputs[i].text
    interface -- kept unmodified so the rest of the pipeline (parsing,
    exact-match scoring, leak detection, saving) needs no changes."""
    def __init__(self, text):
        self.text = text


class _FakeRequestOutput:
    def __init__(self, texts):
        self.outputs = [_FakeCompletion(t) for t in texts]


# Which samples are ALREADY known to verify, and what text we have for each.
#
# These are two separate questions, and conflating them cost real money once:
# an earlier version only learned "already matched" from PREV_RESULTS_PATH and
# only in the branch where the checkpoint was absent. Resuming from a mid-run
# checkpoint therefore threw that knowledge away, and since a seeded sample
# stores just its single winning attempt, every one of them looked "short of
# N_SAMPLES" and got re-queued -- 4558 needless calls across 2279 samples that
# had already matched gold. So: read the matched set from the previous
# completed run whenever that file exists, independently of where the texts
# come from, and let the checkpoint supply texts on top.
#
# Note it reads `verified` (strict OR PA) and falls back to
# `exact_program_match` for files written before that field existed.
texts_by_train_idx = {}
already_matched = set()

if PREV_RESULTS_PATH.exists():
    try:
        prev = json.loads(PREV_RESULTS_PATH.read_text(encoding="utf-8"))
        for r in prev["results"]:
            ok = r.get("verified", r.get("exact_program_match", False))
            if ok and r.get("raw_output"):
                already_matched.add(r["train_index"])
                texts_by_train_idx.setdefault(r["train_index"], [r["raw_output"]])
        print(f"Seeded {len(already_matched)} already-verified samples from "
              f"{PREV_RESULTS_PATH.name} (0 new calls needed for those).")
    except (json.JSONDecodeError, KeyError) as e:
        print(f"{PREV_RESULTS_PATH.name} exists but couldn't be read ({e!r}) -- ignoring it.")

if GEN_CKPT_PATH.exists():
    ckpt = {int(k): v for k, v in json.loads(GEN_CKPT_PATH.read_text(encoding="utf-8")).items()}
    texts_by_train_idx.update(ckpt)
    print(f"Resumed texts for {len(ckpt)} samples from this run's own checkpoint "
          f"({GEN_CKPT_PATH.name}).")

# Failures from a PRIOR prompt version are deliberately NOT reused as attempts:
# only the match/no-match outcome is prompt-independent, a failed attempt's text
# reflects whichever prompt produced it.

# How many MORE attempts each sample still needs.
work = []  # one entry (sample dict) per call still needed
for s in samples:
    idx = s["train_index"]
    have = len(texts_by_train_idx.get(idx, []))
    needed = 0 if idx in already_matched else max(0, N_SAMPLES - have)
    work.extend([s] * needed)

print(f"{len(work)} new API calls needed across "
      f"{len(set(s['train_index'] for s in work)) if work else 0} samples "
      f"({len(already_matched)} samples need 0 -- already matched).")

futures = {}
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    for s in work:
        futures[pool.submit(call_teacher_once, s)] = s["train_index"]

    completed = 0
    for future in tqdm(as_completed(futures), total=len(futures), desc="Generating traces..."):
        idx = futures[future]
        texts_by_train_idx.setdefault(idx, []).append(future.result())
        completed += 1
        if completed % 100 == 0:
            GEN_CKPT_PATH.write_text(json.dumps(texts_by_train_idx, ensure_ascii=False), encoding="utf-8")

GEN_CKPT_PATH.write_text(json.dumps(texts_by_train_idx, ensure_ascii=False), encoding="utf-8")

outputs = [_FakeRequestOutput(texts_by_train_idx[s["train_index"]]) for s in samples]
print(f"Ready. {len(outputs)} samples, "
      f"{sum(len(v) for v in texts_by_train_idx.values())} total attempts across all samples.")


## Parse + validate each trace

Same two checks as the CoNR notebook:
1. `<program>` block must EXACT-match the gold program (same operators/args/
   order/#N refs) via the existing `parse_program`/`compute_program_accuracy`
   logic used across the other notebooks -- not just execution-equivalent.
   This is now the real yield signal, since the teacher genuinely had to
   derive the program itself.
2. `<think>` block must not contain leak phrases -- kept as a defensive check
   even though nothing was revealed this time, in case the model hallucinates
   having been given an answer.

When `N_SAMPLES > 1`, each sample has multiple independent attempts; we keep
the first attempt that exact-matches gold (if any), else the first attempt
(for review), and record how many attempts it took.

In [ ]:
"""
Parser + evaluation utilities for FinQA/VLSP-2025 style computation programs.
(Same implementation as the 0-shot/1-shot/few-shot/SFT notebooks, with one fix:
parse_program now handles nested parentheses inside arguments, e.g. table column
names like "Bien LN HDSXKD (%)" or "P/E (x)" -- the original regex-based
`([a-zA-Z_]+)\\(([^()]*)\\)` couldn't match across a paren nested inside the
argument list, so any program referencing such a column raised ValueError and
got scored as a false mismatch even when generated == gold verbatim. Found via
manual review of the first 150-sample trial run: 9 of 12 "mismatches" were
actually this bug, not a real generation difference.)
"""

from typing import List, Tuple, Optional

VALID_OPERATORS = {
    "add", "subtract", "multiply", "divide", "exp", "greater",
    "table_sum", "table_average", "table_max", "table_min",
}
_NUM_RE = re.compile(r"^-?\d+(\.\d+)?$")
_REF_RE = re.compile(r"^#(\d+)$")


def _split_top_level(s: str, sep: str = ",") -> List[str]:
    parts, depth, current = [], 0, []
    for ch in s:
        if ch == "(":
            depth += 1; current.append(ch)
        elif ch == ")":
            depth -= 1; current.append(ch)
        elif ch == sep and depth == 0:
            parts.append("".join(current)); current = []
        else:
            current.append(ch)
    if current:
        parts.append("".join(current))
    return [p.strip() for p in parts if p.strip() != ""]


def parse_program(program_str: str) -> List[Tuple[str, List[str]]]:
    program_str = program_str.strip().rstrip(",").strip()
    if not program_str:
        raise ValueError("Empty program string.")

    steps: List[Tuple[str, List[str]]] = []
    name_pattern = re.compile(r"\s*([a-zA-Z_]+)\(")
    pos = 0
    text = program_str
    while pos < len(text):
        m = name_pattern.match(text, pos)
        if not m:
            raise ValueError(f"Malformed program near: '{text[pos:pos+30]}...'")
        op = m.group(1).strip()
        if op not in VALID_OPERATORS:
            raise ValueError(f"Unknown operator: '{op}'")

        # Find the matching close paren by depth-counting, so parens nested
        # inside an argument (e.g. a column name like "ROE (%)") don't
        # prematurely end the call.
        start = m.end() - 1  # index of the opening '('
        depth = 0
        i = start
        while i < len(text):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    break
            i += 1
        else:
            raise ValueError(f"Unbalanced parens near: '{text[start:start+30]}...'")

        args = _split_top_level(text[start + 1:i], sep=",")
        steps.append((op, args))
        pos = i + 1
        if pos < len(text) and text[pos] == ",":
            pos += 1

    if not steps:
        raise ValueError("No valid steps parsed.")
    return steps


def _normalize_program(program_str: str) -> List[Tuple[str, Tuple[str, ...]]]:
    steps = parse_program(program_str)
    normalized = []
    for op, args in steps:
        norm_args = []
        for a in args:
            a = a.strip()
            if _REF_RE.match(a) or a.lower() == "none":
                norm_args.append(a.lower())
            else:
                # A trailing '%' means the value IS a percentage, so it must be
                # divided by 100 -- merely stripping the sign treats "15.1%" as
                # 15.1, which is 100x off and silently accepts a teacher program
                # that the repo's official PA scorer would reject. (Measured on
                # the valid split: 2 samples slipped through this way, e.g. gold
                # subtract(15.1%, 15.7%) vs teacher subtract(15.1, 15.7).)
                # This now matches _parse_numeric_literal in the eval notebooks.
                cleaned = a.replace(",", "").replace("$", "").strip()
                is_percent = cleaned.endswith("%")
                if is_percent:
                    cleaned = cleaned[:-1].strip()
                try:
                    value = float(cleaned)
                    if is_percent:
                        value /= 100.0
                    norm_args.append(f"{round(value, 6)}")
                except ValueError:
                    norm_args.append(a)
        normalized.append((op, tuple(norm_args)))
    return normalized


def is_exact_program_match(generated_program: str, gold_program: str) -> bool:
    """Exact structural match (not just execution-equivalent) -- if the teacher
    reformatted/simplified the gold program, we want to know and discard it."""
    try:
        return _normalize_program(generated_program) == _normalize_program(gold_program)
    except ValueError:
        return False


# Regex patterns (not plain substrings) so common, legitimate phrasing like
# "van ban cho biet ..." / "văn bản cho biết ..." (citing what the source
# document says) doesn't false-positive. The first 150-sample trial run used
# plain substrings ("cho biết", "được cho") and flagged 6 samples, of which
# only 4 were real leaks -- 2 were the model citing pre_text/table content,
# which is expected and fine. These patterns require "chương trình"/"kết
# quả"/"đáp án" immediately followed by "được cho" to actually catch the
# forbidden pattern (the teacher referring to the program/answer itself as
# something it was handed), not just any use of "cho biết"/"được cho".
LEAK_PATTERNS = [
    r"chương trình\s+được cho",
    r"kết quả\s+được cho",
    r"đáp án\s+được cho",
    r"theo chương trình được cho",
    r"đã biết trước",
    r"\bverified\b",
    r"\bas given\b",
    r"\bwe are told\b",
    r"\bgiven answer\b",
    r"\bgiven program\b",
]


def find_leak_phrases(think_block: str) -> List[str]:
    lowered = think_block.lower()
    return [pat for pat in LEAK_PATTERNS if re.search(pat, lowered)]


def extract_think_and_program(raw_text: str):
    """
    Extract the reasoning trace and program string from the model's raw output.

    Qwen3-Next-Thinking doesn't emit a literal opening `<think>` tag -- the
    chat template already puts the model in "thinking" mode at the start of
    the assistant turn, so the decoded text starts directly with the
    reasoning content and only closes with `</think>`. The first 150-sample
    trial run showed this is 100% consistent (0/150 had an opening tag), which
    made every `think` come back None under the original `<think>(.*?)</think>`
    regex. So: treat everything before the first `</think>` as the reasoning
    block (whether or not an opening tag happens to be present), and extract
    `<program>...</program>` as before.
    """
    close_think_idx = raw_text.find("</think>")
    if close_think_idx == -1:
        think = None
    else:
        think = raw_text[:close_think_idx]
        think = re.sub(r"^\s*<think>\s*", "", think)  # strip an opening tag if present
        think = think.strip()

    program_match = re.search(r"<program>(.*?)</program>", raw_text, re.DOTALL)
    program = program_match.group(1).strip() if program_match else None

    return think, program

In [ ]:
# --- Official PA scorer, used ALONGSIDE is_exact_program_match ---------------
# The two are complementary tests of "did the teacher derive the same
# computation", and neither contains the other. Measured over the 2993 train
# samples of this run:
#   strict only (74) : normalises the number's surface form, so `100` == `100.00`
#                      and `20%` == `0.2`, but demands identical step structure.
#   PA only     (28) : sympy-simplifies, so commuted arguments and reordered
#                      steps pass, but literals are opaque strings, so PA
#                      wrongly rejects `100` vs `100.00`.
# Accepting either lifts verified coverage 2334 -> 2362 with zero new API calls.
# Both sets were inspected for false positives: the 28 PA-only recoveries are
# all genuine commutations/step reorderings, the 74 strict-only are all cosmetic
# number formatting where the teacher's arithmetic equals gold's exactly.
#
# This filter is a DIFFERENT job from grading the student, so the stricter
# metric is not automatically the safer choice here: the teacher's program is
# thrown away once it has vouched for the trace, because `qa.program` in the
# final file stays the untouched gold program. The filter only has to answer
# "was the teacher's reasoning right", and how it spelled `100` says nothing
# about that.
import sys

_SCORER_DIRS = [Path("notebooks/evaluate"), Path("../../evaluate")]
_scorer_dir = next((p for p in _SCORER_DIRS if (p / "scorer.py").exists()), None)
if _scorer_dir is None:
    raise FileNotFoundError(
        f"Could not find scorer.py. cwd={Path.cwd()}, "
        f"tried: {[str(p) for p in _SCORER_DIRS]}"
    )
sys.path.insert(0, str(_scorer_dir.resolve()))
from scorer import program_tokenization, equal_program


def is_pa_match(generated_program: str, gold_program: str) -> bool:
    """Official Program Accuracy: sympy symbolic equivalence, gold vs prediction."""
    if not generated_program:
        return False
    try:
        return equal_program(program_tokenization(gold_program),
                             program_tokenization(generated_program))
    except Exception:
        return False


def is_verified(generated_program: str, gold_program: str) -> bool:
    """Trust the trace if EITHER correctness test accepts its program."""
    return (is_exact_program_match(generated_program, gold_program)
            or is_pa_match(generated_program, gold_program))


print("PA scorer loaded from", _scorer_dir.resolve())


In [ ]:
results = []
n_verified = 0
n_strict = 0
n_pa = 0
n_missing_tags = 0
n_leak = 0

for s, output in zip(samples, outputs):
    chosen_raw = chosen_think = chosen_program = None
    chosen_verified = False
    attempts_used = 0
    # Try each independent attempt in order; stop at the first one that
    # verifies against gold. If none do, keep the first attempt for review.
    for attempt_idx, cand in enumerate(output.outputs, start=1):
        raw_text = cand.text
        think, program = extract_think_and_program(raw_text)
        verified = is_verified(program, s["program"]) if program else False
        if chosen_raw is None:
            chosen_raw, chosen_think, chosen_program, attempts_used = raw_text, think, program, attempt_idx
        if verified:
            chosen_raw, chosen_think, chosen_program, attempts_used = raw_text, think, program, attempt_idx
            chosen_verified = True
            break

    leaks = find_leak_phrases(chosen_think) if chosen_think else []
    strict_ok = is_exact_program_match(chosen_program, s["program"]) if chosen_program else False
    pa_ok = is_pa_match(chosen_program, s["program"]) if chosen_program else False

    if chosen_think is None or chosen_program is None:
        n_missing_tags += 1
    if chosen_verified:
        n_verified += 1
    n_strict += int(strict_ok)
    n_pa += int(pa_ok)
    if leaks:
        n_leak += 1

    results.append({
        "train_index": s["train_index"],
        "evidence_type": s["evidence_type"],
        "question": s["question"],
        "gold_program": s["program"],
        "gold_answer": s["answer"],
        "attempts_used": attempts_used,
        "raw_output": chosen_raw,
        "parsed_think": chosen_think,
        "parsed_program": chosen_program,
        # `verified` is what every downstream cell filters on. The two
        # component tests are kept separately because they disagree in both
        # directions and the split is worth being able to audit later.
        "verified": chosen_verified,
        "exact_program_match": strict_ok,
        "pa_match": pa_ok,
        # How this trace was produced. "independent" = teacher saw only the
        # context and question. The rationalization pass below overwrites this
        # with "answer_conditioned" for the samples it rescues, so the two can
        # always be separated -- and ablated against each other -- downstream.
        "trace_source": "independent",
        "leak_phrases_found": leaks,
    })

n = len(results)
# Defined here too, not only in the rationalization cell below, so that the
# save/export cells still work if that pass is skipped. At this point every
# verified trace came from the independent solve by definition.
n_independent = n_verified
n_conditioned = 0

print(f"Total: {n}")
print(f"VERIFIED (strict OR PA) : {n_verified} ({100 * n_verified / n:.1f}%)   <- what gets kept")
print(f"  strict only component : {n_strict} ({100 * n_strict / n:.1f}%)")
print(f"  PA only component     : {n_pa} ({100 * n_pa / n:.1f}%)")
print(f"Missing <think>/<program> tags: {n_missing_tags}")
print(f"Traces with leak phrases: {n_leak}")

# --- did the convention rules land? --------------------------------------
# v1 (no convention rules) scored 14.4% exact match on the full train split,
# and its misses were dominated by four fixable formatting habits. Counting
# them here says whether each rule is being obeyed, which is far more
# actionable than the headline yield alone.
import re as _re

_V1_EXACT_RATE = 14.4
_checks = {
    "R1 nested call in args": lambda p: bool(_re.search(r"\((?:[^()]*)\b[a-zA-Z_]+\(", p)),
    "R2 trailing multiply(#N, 100)": lambda p: bool(_re.search(r"multiply\(#\d+,\s*100\)\s*$", p)),
    "R3 invented operator": lambda p: any(
        o not in VALID_OPERATORS for o in _re.findall(r"([a-zA-Z_]+)\(", p)),
    "R5 table_*() around one value": lambda p: bool(
        _re.search(r"table_(?:sum|average|max|min)\(\s*[^,()]+\s*\)", p)),
}
_progs = [r["parsed_program"] for r in results if r["parsed_program"]]
print(f"\nRule violations among {len(_progs)} parsed programs:")
for _name, _fn in _checks.items():
    _n = sum(1 for p in _progs if _fn(p))
    print(f"  {_name:32s} {_n:4d} ({100 * _n / max(len(_progs), 1):.1f}%)")

_rate = 100 * n_verified / n
print(f"\nVerified {_rate:.1f}% vs v1 baseline {_V1_EXACT_RATE}% "
      f"-> {'BETTER' if _rate > _V1_EXACT_RATE else 'NOT better'}")


## Rationalization pass (STaR step 4) for the samples that stayed unverified

The independent solve tops out around 79%, and an error analysis of the residue
showed why further prompt engineering will not fix it: the largest failure
class is **not** a reasoning error but an unresolvable labelling ambiguity.
Over the 278 train questions phrased "X tăng/giảm bao nhiêu", 53.6% of gold
programs end in a `divide` (relative change) and 46.4% do not (absolute
change) — the wording does not predict which, so no rule can. The same trap
already cost one regression when the `multiply(#N, 100)` rule was written off
a small sample and over-applied at scale (80% → 64%).

What *does* settle it is the answer itself: knowing the result is `0.0699`
rather than `90` determines the convention immediately.

So this pass follows the rationalization step of STaR (Zelikman et al., 2022):
for the failures only, supply the gold `exe_ans` as a hint, regenerate, and
keep the trace only if its program now verifies. Note this borrows **one step**
of STaR, not its outer loop — STaR loops because the model being trained is the
one generating, so each round it gets stronger. Here the teacher is a frozen
external model, so re-running it only resamples (which is what `N_SAMPLES=3`
already did, with diminishing returns). This is a single pass.

**The hint goes to the teacher, never to the student.** The student's training
contract is unchanged: input is context + question with no answer, and
`qa.program` stays the untouched gold program.

Safeguards, because the real risk is a trace that silently reasons backwards:

1. Only `exe_ans` is supplied — never the gold program. The teacher still has
   to find the computation; the answer only settles which convention to use.
2. Rule `H1` forbids referring to the answer at all.
3. Leak detection is **calibrated, not guessed**: across the 2334 traces the
   independent pass wrote without the answer, `"đáp án"` occurred **0 times
   (0.00%)**, so any occurrence now is a genuine leak. `"kết quả cuối"` occurred
   192 times (8.23%) and is deliberately *not* flagged — it is normal phrasing.
4. A leaking trace is **rejected**, not merely counted.
5. Program verification is unchanged — a trace whose program still disagrees
   with gold is discarded no matter how good it reads.
6. Rescued samples are tagged `trace_source="answer_conditioned"`, so an SFT
   run can train with and without them and *measure* whether they help, rather
   than trusting the argument above.

Report the two numbers separately: 78.9% is the *independent-solve* rate, and
whatever this reaches is *trace coverage*. Conflating them would overstate the
teacher.


In [ ]:
N_RATIONALIZE = 2   # attempts per still-unverified sample
RAT_CKPT_PATH = OUTPUT_DIR / f"rationalize_ckpt_{TRAIN_JSON_PATH.stem}_n{N_RATIONALIZE}.json"

# The base prompt, plus the hint section. Everything else (R1-R9, S1-S4) is
# inherited unchanged, so the only difference from the independent pass is the
# supplied answer and the rules governing it.
CONR_SYSTEM_PROMPT_HINTED = CONR_SYSTEM_PROMPT.replace(
    "### OUTPUT FORMAT (strict):",
    """### THE FINAL NUMERIC ANSWER IS SUPPLIED THIS TIME

You are given the question's final numeric answer. It is supplied for ONE
reason: this dataset is inconsistent about labelling conventions that the
question's wording genuinely cannot settle, as measured over its own gold
programs --
  * whether "X tang/giam bao nhieu" wants the absolute difference or the
    relative change (53.6% of such gold programs divide by the base, 46.4%
    do not -- the wording does not predict which), and
  * which unit scale to read the numbers at (ty vs trieu vs raw digits).
Use the answer ONLY to settle choices like these. Derive the whole computation
yourself exactly as before; do not work backwards from the number.

H1. Your <think> block must read exactly as if the answer had NOT been given.
    Never mention it. Never write "dap an", "gia tri dich", "khop voi",
    "bang dung", "de ra duoc", and never reason in the form "since the answer
    is X, therefore ...". Justify every step from the context and the question
    alone.
      WRONG: "vi dap an la 0.0699 nen ta phai chia cho gia tri goc"
      RIGHT: "vi cau hoi hoi muc tang tuong doi nen ta chia chenh lech cho
              gia tri nam goc"
H2. The program must actually evaluate to the supplied answer. If your first
    derivation does not reach it, reconsider which values or which operation
    you chose BEFORE writing -- but the <think> block must still present one
    clean derivation with no trace of that reconsideration (S3 still applies).
    Never bolt on an artificial step just to force the number out.

### OUTPUT FORMAT (strict):""")

CONR_USER_FRAME_HINTED = CONR_USER_FRAME.replace(
    "### YOUR TASK:",
    """### FINAL NUMERIC ANSWER (to disambiguate conventions only -- never mention it):
{gold_answer}

### YOUR TASK:""")

# Calibrated against the 2334 traces the independent pass wrote WITHOUT the
# answer: "dap an" occurred 0 times (0.00%) there, so any occurrence now is a
# real leak rather than normal phrasing. "ket qua cuoi" occurred 192 times
# (8.23%) and is deliberately absent from this list -- flagging it would reject
# legitimate traces wholesale.
#
# Patterns are written WITHOUT diacritics and matched against diacritic-folded
# text, because an accent-sensitive matcher misses the most likely leak of all:
# rule H1 above spells its own counter-example unaccented ("vi dap an la
# 0.0699"), so a model echoing that phrasing would have slipped straight past a
# matcher keyed on "dap an" with diacritics. Folding both sides catches the
# leak whether or not the model writes accents.
import unicodedata

HINT_LEAK_PATTERNS = [
    r"dap an",
    r"gia tri dich",
    r"bang dung",
    r"de ra duoc",
    r"khop voi",
    r"\bthe (given|target|final) (answer|value)\b",
    r"\bgiven (answer|value|result)\b",
    r"\bto match the\b",
    r"\bworking backwards?\b",
]


def _fold(text):
    """Lowercase and strip Vietnamese diacritics ('d' is not decomposed by NFD)."""
    decomposed = unicodedata.normalize("NFD", text.lower())
    return "".join(c for c in decomposed if unicodedata.category(c) != "Mn").replace("đ", "d")


def find_hint_leaks(think_block):
    folded = _fold(think_block)
    return [p for p in LEAK_PATTERNS + HINT_LEAK_PATTERNS if re.search(_fold(p), folded)]


def _numbers_in(text):
    """Every number in `text`, read under both the '.' and ',' decimal conventions."""
    vals = set()
    for tok in re.findall(r"-?\d[\d.,]*", text):
        tok = tok.rstrip(".,")
        for cand in {tok.replace(",", "."), tok.replace(".", "").replace(",", "."),
                     tok.replace(",", ""), tok.replace(".", "")}:
            try:
                vals.add(float(cand))
            except ValueError:
                pass
    return vals


def _program_literals(program):
    args = []
    for m in re.finditer(r"[a-zA-Z_]+\(([^()]*(?:\([^()]*\)[^()]*)*)\)", program):
        args += [a.strip() for a in m.group(1).split(",")]
    out = set()
    for a in args:
        a = a.rstrip("%")
        if re.fullmatch(r"-?\d+\.?\d*", a):
            out.add(float(a))
    return out


def echoes_answer(think_block, program, gold_answer):
    """True if the trace states the supplied answer as a bare number.

    The word-based leak detector cannot see a purely numeric echo -- a trace
    that says "ta duoc 0.0919" without ever writing "dap an" reads clean but is
    still reciting the hint. Numbers that the program actually consumes are
    exempt, since quoting an input is exactly what R7 asks for.

    Calibrated on the 2334 traces written WITHOUT the hint: only 1.1% mention
    the answer as a non-input number. So the rule costs ~1% of otherwise-valid
    rescues, which is a cheap price for closing a channel the text patterns miss.
    """
    try:
        target = float(gold_answer)
    except (TypeError, ValueError):
        return False
    tol = max(1e-6, abs(target) * 1e-6)
    if any(abs(v - target) < tol for v in _program_literals(program)):
        return False
    return any(abs(v - target) < tol for v in _numbers_in(think_block))


def call_teacher_hinted(sample_dict, max_retries=3):
    user_msg = CONR_USER_FRAME_HINTED.format(
        pre_text=sample_dict["pre_text"], table=sample_dict["table"],
        post_text=sample_dict["post_text"], question=sample_dict["question"],
        gold_answer=sample_dict["answer"],
    )
    messages = [
        {"role": "system", "content": CONR_SYSTEM_PROMPT_HINTED},
        {"role": "user", "content": user_msg},
    ]
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL, messages=messages,
                temperature=0.6, top_p=0.9, max_tokens=MAX_TOKENS,
            )
            text = resp.choices[0].message.content
            if text and text.strip():
                return text.strip()
        except Exception as e:
            if attempt == max_retries:
                print(f"[attempt failed after {max_retries} tries] {e}")
                return ""
        if attempt < max_retries:
            time.sleep(2 * attempt)
    return ""


todo = [s for s, r in zip(samples, results) if not r["verified"]]

rat_texts = {}
if RAT_CKPT_PATH.exists():
    rat_texts = {int(k): v for k, v in json.loads(RAT_CKPT_PATH.read_text(encoding="utf-8")).items()}
    print(f"Resumed {len(rat_texts)} samples from {RAT_CKPT_PATH.name}.")

work = []
for s in todo:
    needed = max(0, N_RATIONALIZE - len(rat_texts.get(s["train_index"], [])))
    work.extend([s] * needed)

print(f"{len(todo)} samples still unverified after the independent pass.")
print(f"{len(work)} new API calls needed ({N_RATIONALIZE} attempts each).")

if work:
    futures = {}
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        for s in work:
            futures[pool.submit(call_teacher_hinted, s)] = s["train_index"]
        completed = 0
        for future in tqdm(as_completed(futures), total=len(futures), desc="Rationalizing..."):
            rat_texts.setdefault(futures[future], []).append(future.result())
            completed += 1
            if completed % 100 == 0:
                RAT_CKPT_PATH.write_text(json.dumps(rat_texts, ensure_ascii=False), encoding="utf-8")
    RAT_CKPT_PATH.write_text(json.dumps(rat_texts, ensure_ascii=False), encoding="utf-8")

# Accept a rescued trace only if it verifies AND does not leak the hint.
idx_of = {r["train_index"]: i for i, r in enumerate(results)}
accepted, rejected_leak, still_wrong = 0, 0, 0

for s in todo:
    ti = s["train_index"]
    took = False
    leaked_here = False
    for raw_text in rat_texts.get(ti, []):
        think, program = extract_think_and_program(raw_text)
        if not think or not program:
            continue
        if not is_verified(program, s["program"]):
            continue
        if find_hint_leaks(think):          # named the answer in words
            leaked_here = True
            continue
        if echoes_answer(think, program, s["answer"]):   # recited it as a number
            leaked_here = True
            continue
        results[idx_of[ti]].update({
            "raw_output": raw_text,
            "parsed_think": think,
            "parsed_program": program,
            "verified": True,
            "exact_program_match": is_exact_program_match(program, s["program"]),
            "pa_match": is_pa_match(program, s["program"]),
            "trace_source": "answer_conditioned",
            "leak_phrases_found": [],
        })
        accepted += 1
        took = True
        break
    if not took:
        if leaked_here:
            rejected_leak += 1
        else:
            still_wrong += 1

n = len(results)
n_verified = sum(1 for r in results if r["verified"])
n_independent = sum(1 for r in results if r["verified"] and r["trace_source"] == "independent")
n_conditioned = sum(1 for r in results if r["verified"] and r["trace_source"] == "answer_conditioned")

print(f"\nRescued by rationalization : {accepted}")
print(f"Rejected (verified but leaked the hint, in words or as a bare number) : {rejected_leak}")
print(f"Still unverified : {still_wrong}")
print(f"\n{'INDEPENDENT-SOLVE rate':<26}: {n_independent} / {n} ({100 * n_independent / n:.1f}%)"
      "   <- the teacher's real ability, report this as the teacher metric")
print(f"{'TRACE COVERAGE':<26}: {n_verified} / {n} ({100 * n_verified / n:.1f}%)"
      "   <- how much of the split carries a trace")
print(f"{'  of which answer-conditioned':<26}: {n_conditioned} ({100 * n_conditioned / n:.1f}%)")

# Eyeball a couple of rescued traces -- automated leak detection catches the
# blatant cases, but only reading them catches a trace that is subtly post-hoc.
_rescued = [r for r in results if r["trace_source"] == "answer_conditioned"][:2]
for r in _rescued:
    print("\n" + "=" * 70)
    print("Q:", r["question"][:150])
    print("gold:", r["gold_program"], "| answer:", r["gold_answer"])
    print("pred:", r["parsed_program"])
    print("trace:", (r["parsed_think"] or "")[:600])


In [ ]:
import csv

# Separate, easy-to-scan file for JUST the failures -- gold vs parsed program
# side by side, so you can eyeball where gemma actually goes wrong (wrong
# number picked, wrong operator, malformed program, etc.) without wading
# through the full JSON. These are the samples that survived BOTH the
# independent solve and the rationalization pass unverified.
fail_path = OUTPUT_DIR / f"independent_solve_trace_{TRAIN_JSON_PATH.stem}_FAILURES.csv"
failures = [r for r in results if not r["verified"]]

with open(fail_path, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "train_index", "evidence_type", "question",
        "gold_program", "gold_answer", "parsed_program", "attempts_used",
        "missing_program_or_think", "leak_phrases_found", "parsed_think",
    ])
    writer.writeheader()
    for r in failures:
        writer.writerow({
            "train_index": r["train_index"],
            "evidence_type": r["evidence_type"],
            "question": r["question"],
            "gold_program": r["gold_program"],
            "gold_answer": r["gold_answer"],
            "parsed_program": r["parsed_program"] or "",
            "attempts_used": r["attempts_used"],
            "missing_program_or_think": r["parsed_think"] is None or r["parsed_program"] is None,
            "leak_phrases_found": ", ".join(r["leak_phrases_found"]),
            "parsed_think": (r["parsed_think"] or "")[:1000],
        })

print(f"Saved {len(failures)} failure cases to {fail_path}")
print("Columns: gold_program vs parsed_program side-by-side, plus the reasoning "
      "trace (truncated to 1000 chars) so you can eyeball where it went wrong. "
      "utf-8-sig encoding so Excel opens the Vietnamese text correctly.")


## Save all results for review

Saves every sample's raw output + parsed fields + validation flags, regardless
of pass/fail, so the full set can be read through to judge trace quality and
yield before scaling `SAMPLE_LIMIT` up (or removing it) for a full run.

In [ ]:
# Named after the input split (e.g. train.json -> ..._train.json) so
# running this notebook twice (once per split) doesn't overwrite the other
# split's results.
output_path = OUTPUT_DIR / f"independent_solve_trace_{TRAIN_JSON_PATH.stem}.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump({
        "model": MODEL,
        "n_samples": len(results),
        "summary": {
            # Kept apart on purpose: the first is what the teacher can do
            # unaided, the second is how much of the split ends up with a
            # trace. Reporting the second as if it were the first would
            # overstate the teacher.
            "independent_solve_verified": n_independent,
            "trace_coverage_verified": n_verified,
            "answer_conditioned": n_conditioned,
            "strict_component": sum(1 for r in results if r["exact_program_match"]),
            "pa_component": sum(1 for r in results if r["pa_match"]),
            "missing_tags": n_missing_tags,
            "leak_phrases": n_leak,
        },
        "results": results,
    }, f, ensure_ascii=False, indent=2)

print(f"Saved {len(results)} results to {output_path}")


In [16]:
# Build the training-ready files. Same schema as
# datasets/ViNumQA/{train,valid}_with_reasoning_trace.json (pre_text/table/
# post_text/id/qa, with qa.reasoning_trace added).
#
# `qa.program` is the UNTOUCHED gold program in both files, never the teacher's.
# The teacher's program only ever served as evidence that its trace could be
# trusted, and is discarded here.
#
# Two files, differing only in how strictly a trace had to be vouched for:
#
# Both files are restricted to trace_source == "independent" -- the
# rationalization pass (hint = gold exe_ans) is NOT used here. Read on
# `results` in the kernel, its rescued samples are simply not selected below;
# nothing needs re-running to change this choice.
#
# Why: manually reading a sample of the rescued traces showed roughly half
# were not fixing a real reasoning error but papering over the dataset's own
# inconsistent labelling -- e.g. gold sometimes reports a relative change
# (subtract then divide by the base) and sometimes an absolute one for
# identically-phrased questions, and the hint just tells the teacher which
# way gold went this time. A few rescued traces were even asked to defend a
# WORSE answer than the teacher's unaided one (its unaided attempt used
# table_sum(row, none) per rule R4; gold enumerated the row by hand, so the
# rescued trace has to rationalize abandoning the correct convention). That
# risk is worse than the traces it prevents from having an empty <think>.
#
#   _pa  : the official PA scorer alone. Strictest.
#   (default) : PA OR is_exact_program_match (the union; also accepts a
#          program whose arithmetic equals gold but whose number formatting
#          differs, e.g. `100.00` vs `100`, which PA rejects on a string
#          comparison even though the arithmetic is identical).
#
# qa.trace_source is kept in both files ("independent" vs "answer_conditioned")
# so a later run can still split on it without regenerating anything.
def build_samples(keep):
    out = []
    for r in results:
        if not keep(r):
            continue
        original = train_data[r["train_index"]]
        out.append({
            "pre_text": original["pre_text"],
            "table": original["table"],
            "post_text": original["post_text"],
            "id": original["id"],
            "qa": {
                "question": original["qa"]["question"],
                "program": original["qa"]["program"],
                "exe_ans": original["qa"]["exe_ans"],
                "reasoning_trace": r["parsed_think"],
                "trace_source": r["trace_source"],
            },
        })
    return out


stem = TRAIN_JSON_PATH.stem
for suffix, keep in [("_pa", lambda r: r["pa_match"] and r["trace_source"] == "independent"),
                     ("",    lambda r: r["verified"] and r["trace_source"] == "independent")]:
    samples_out = build_samples(keep)
    path = OUTPUT_DIR / f"{stem}_with_reasoning_trace{suffix}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(samples_out, f, ensure_ascii=False, indent=2)
    cond = sum(1 for s in samples_out if s["qa"]["trace_source"] == "answer_conditioned")
    label = "PA + independent" if suffix else "union + independent"
    print(f"{label:<16} {len(samples_out):>5} / {len(results)} "
          f"({100 * len(samples_out) / len(results):.1f}%)  "
          f"answer_conditioned={cond}  ->  {path.name}")

print(f"\nSaved to {OUTPUT_DIR}")
print("NOTE: both files contain ONLY the kept samples. The v5-style packaging "
      "(every sample present, reasoning_trace=null where not kept, so the SFT "
      "run loses no training data) is still a separate step.")


PA + independent   455 / 584 (77.9%)  answer_conditioned=0  ->  valid_with_reasoning_trace_pa.json
union + independent   470 / 584 (80.5%)  answer_conditioned=0  ->  valid_with_reasoning_trace.json

Saved to outputs\conr_trace_gemma_independent_solve
NOTE: both files contain ONLY the kept samples. The v5-style packaging (every sample present, reasoning_trace=null where not kept, so the SFT run loses no training data) is still a separate step.
